In [1]:
name = 'PTT'

In [2]:
import json
import os
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine

# Database connection details
db_user = 'root'
db_password = ''
db_host = 'localhost'
db_name = 'stock'
table_name = 'price'

# Create a connection to the database
engine = create_engine("mysql+pymysql://root:@localhost:3306/stock")
const = engine.connect()

In [3]:
# Get the user's home directory
user_path = os.path.expanduser('~')
# Get the current working directory
current_path = os.getcwd()
# Derive the base directory (base_dir) by removing the last folder ('Daily')
base_path = os.path.dirname(current_path)
#C:\Users\PC1\OneDrive\A5\Data
dat_path = os.path.join(base_path, "Data")
#C:\Users\PC1\OneDrive\Imports\santisoontarinka@gmail.com - Google Drive\Data>
god_path = os.path.join(user_path, "OneDrive","Imports","santisoontarinka@gmail.com - Google Drive","Data")
#C:\Users\PC1\iCloudDrive\data
icd_path = os.path.join(user_path, "iCloudDrive", "Data")
#C:\Users\PC1\OneDrive\Documents\obsidian-git-sync\Data
osd_path = os.path.join(user_path, "OneDrive","Documents","obsidian-git-sync","Data")
#C:\Users\PC1\OneDrive\A5\Excel
csv_path = os.path.join(base_path, "CSV")

In [4]:
print("User path:", user_path)
print(f"Current path: {current_path}")
print(f"Base path: {base_path}")
print(f"Data path (dat_path): {dat_path}") 
print(f"CSV path (csv_path): {csv_path}") 
print(f"Google Drive path (god_path): {god_path}")
print(f"iCloudDrive path (icd_path): {icd_path}") 
print(f"Obsidian path (osd_path): {osd_path}")

User path: C:\Users\PC1
Current path: C:\Users\PC1\OneDrive\A4\Database
Base path: C:\Users\PC1\OneDrive\A4
Data path (dat_path): C:\Users\PC1\OneDrive\A4\Data
CSV path (csv_path): C:\Users\PC1\OneDrive\A4\CSV
Google Drive path (god_path): C:\Users\PC1\OneDrive\Imports\santisoontarinka@gmail.com - Google Drive\Data
iCloudDrive path (icd_path): C:\Users\PC1\iCloudDrive\Data
Obsidian path (osd_path): C:\Users\PC1\OneDrive\Documents\obsidian-git-sync\Data


In [5]:
# Read the table into a Pandas DataFrame
df = pd.read_sql(f'SELECT * FROM price', con=const)
df.shape

(398002, 7)

In [6]:
sql = f"""SELECT * 
FROM price 
WHERE name = '{name}' 
  AND date >= CURDATE() - INTERVAL 5 YEAR 
ORDER BY date DESC
"""
print(sql)

SELECT * 
FROM price 
WHERE name = 'PTT' 
  AND date >= CURDATE() - INTERVAL 5 YEAR 
ORDER BY date DESC



In [7]:
df = pd.read_sql(sql, con=const)
df.shape

(1207, 7)

In [8]:
file_name = name + '.csv'
output_file = os.path.join(dat_path, file_name)
god_file = os.path.join(god_path, file_name)
icd_file = os.path.join(icd_path, file_name)
osd_file = os.path.join(osd_path, file_name)
csv_file = os.path.join(csv_path, file_name)

print(f"Output file : {output_file}") 
print(f"icd_file : {icd_file}") 
print(f"god_file : {god_file}") 
print(f"osd_file : {osd_file}") 
print(f"csv_file : {csv_file}") 

Output file : C:\Users\PC1\OneDrive\A4\Data\PTT.csv
icd_file : C:\Users\PC1\iCloudDrive\Data\PTT.csv
god_file : C:\Users\PC1\OneDrive\Imports\santisoontarinka@gmail.com - Google Drive\Data\PTT.csv
osd_file : C:\Users\PC1\OneDrive\Documents\obsidian-git-sync\Data\PTT.csv
csv_file : C:\Users\PC1\OneDrive\A4\CSV\PTT.csv


In [9]:
df.sort_values(['date'],ascending=[True]).to_csv(output_file, index=False)
df.sort_values(['date'],ascending=[True]).to_csv(god_file, index=False)
df.sort_values(['date'],ascending=[True]).to_csv(osd_file, index=False)
df.sort_values(['date'],ascending=[True]).to_csv(icd_file, index=False)
df.sort_values(['date'],ascending=[True]).to_csv(csv_file, index=False)

In [10]:
# Complete cell to generate standalone HTML with embedded data

# stock name and data already loaded in df (from previous cells)
# Ensure df is sorted by date ascending and has the last 5 years of data
# If df is empty, re-run the SQL query cell

# Prepare data for JavaScript with explicit numeric conversion
data_for_js = []
for _, row in df.iterrows():
    # Convert date to string in YYYY-MM-DD format
    date_val = row["date"]
    if hasattr(date_val, "strftime"):
        date_str = date_val.strftime("%Y-%m-%d")
    else:
        date_str = str(date_val)
    data_for_js.append({
        "dateStr": date_str,
        "price": float(row["price"]),
        "maxp": float(row["maxp"]),
        "minp": float(row["minp"]),
        "qty": float(row["qty"]),
        "opnp": float(row["opnp"])
    })

# Verify data length
print(f"Data points to embed: {len(data_for_js)}")
if len(data_for_js) == 0:
    raise ValueError("No data to embed. Check your database query.")

# Convert to JSON string with proper escaping
data_json_str = json.dumps(data_for_js, ensure_ascii=False)

# HTML template with embedded data and debug console logs
html_template = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0, user-scalable=yes">
    <title>Stock Vision AI | {symbol} Analysis</title>
    <!-- Plotly + Tailwind CSS -->
    <script src="https://cdn.plot.ly/plotly-3.0.1.min.js" charset="utf-8"></script>
    <script src="https://cdn.tailwindcss.com"></script>
    <style>
        .card-hover:hover {{ transform: translateY(-2px); transition: all 0.2s ease; box-shadow: 0 10px 25px -5px rgba(0,0,0,0.1); }}
        .chart-container {{ transition: all 0.2s; background: #ffffff; border-radius: 20px; padding: 1rem; box-shadow: 0 1px 3px rgba(0,0,0,0.05); }}
        .stats-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(160px, 1fr)); gap: 1rem; }}
        .filter-btn.active {{ background: #1e40af; color: white; border-color: #1e40af; }}
    </style>
</head>
<body class="bg-gradient-to-br from-slate-50 to-gray-100 font-sans antialiased">

    <div class="max-w-7xl mx-auto px-4 py-6 md:py-8">
        <div class="flex flex-wrap justify-between items-center mb-6 gap-3">
            <div>
                <h1 class="text-3xl md:text-4xl font-extrabold bg-gradient-to-r from-slate-800 to-slate-600 bg-clip-text text-transparent">📈 Stock <span class="bg-gradient-to-r from-blue-700 to-indigo-700 bg-clip-text text-transparent">Graphic</span> Analyst</h1>
                <p class="text-gray-500 mt-1 text-sm">Stock: <span class="font-bold text-gray-700">{symbol}</span> | Data: last 5 years | Filters: 1Y · 6M · 3M · 1M · custom days</p>
            </div>
        </div>

        <div class="grid grid-cols-1 lg:grid-cols-3 gap-6">
            <!-- LEFT COLUMN -->
            <div class="lg:col-span-1 space-y-6">
                <div class="bg-white/80 backdrop-blur-sm rounded-2xl shadow-lg p-5 border border-gray-100 card-hover">
                    <h2 class="text-lg font-bold flex items-center gap-2 text-gray-800 border-b pb-2 mb-3"><span class="text-2xl">📊</span> Key Metrics</h2>
                    <div class="stats-grid" id="keyMetricsGrid">
                        <div class="bg-slate-50 p-3 rounded-xl text-center"><p class="text-xs text-gray-500">Latest Price</p><p class="text-2xl font-bold text-blue-700" id="latestPrice">—</p></div>
                        <div class="bg-slate-50 p-3 rounded-xl text-center"><p class="text-xs text-gray-500">Highest (all-time)</p><p class="text-2xl font-bold text-emerald-700" id="highPrice">—</p></div>
                        <div class="bg-slate-50 p-3 rounded-xl text-center"><p class="text-xs text-gray-500">Lowest Price</p><p class="text-2xl font-bold text-rose-700" id="lowPrice">—</p></div>
                        <div class="bg-slate-50 p-3 rounded-xl text-center"><p class="text-xs text-gray-500">Avg Close</p><p class="text-xl font-semibold" id="avgPrice">—</p></div>
                        <div class="bg-slate-50 p-3 rounded-xl text-center"><p class="text-xs text-gray-500">Total Return (%)</p><p class="text-xl font-semibold" id="totalReturn">—</p></div>
                        <div class="bg-slate-50 p-3 rounded-xl text-center"><p class="text-xs text-gray-500">Volatility (annual)</p><p class="text-xl font-semibold" id="volatility">—</p></div>
                        <div class="bg-slate-50 p-3 rounded-xl text-center"><p class="text-xs text-gray-500">Avg Volume (qty)</p><p class="text-xl font-semibold" id="avgVolume">—</p></div>
                        <div class="bg-slate-50 p-3 rounded-xl text-center"><p class="text-xs text-gray-500">Total Volume</p><p class="text-xl font-semibold" id="totalVolume">—</p></div>
                    </div>
                </div>

                <div class="bg-white/80 backdrop-blur-sm rounded-2xl shadow-lg p-5 border border-gray-100">
                    <h2 class="text-lg font-bold flex gap-2 items-center"><span>🧠</span> AI Market Pulse</h2>
                    <div id="insightText" class="text-sm text-gray-700 mt-3 space-y-2"></div>
                    <div class="mt-3 pt-2 border-t border-dashed">
                        <p class="text-xs font-semibold text-gray-500">🔥 Top 3 volume days (filtered period)</p>
                        <ul id="topVolumeDays" class="text-xs text-gray-600 list-disc pl-4 mt-1"></ul>
                    </div>
                </div>

                <div class="bg-white/80 backdrop-blur-sm rounded-2xl shadow-lg p-4 border border-gray-100">
                    <h2 class="text-md font-bold flex gap-2 items-center"><span>🔥</span> Correlation Matrix (filtered data)</h2>
                    <div id="correlationHeatmap" class="h-72 w-full mt-2"></div>
                    <p class="text-[11px] text-gray-400 mt-2 text-center">Pearson correlation among numeric fields</p>
                </div>
            </div>

            <!-- RIGHT COLUMN -->
            <div class="lg:col-span-2 space-y-5">
                <div class="bg-white/80 backdrop-blur-sm rounded-xl shadow-md p-4 flex flex-wrap gap-3 items-center justify-between">
                    <div class="flex flex-wrap gap-2 items-center">
                        <span class="text-sm font-semibold text-gray-700">⏱️ Date filter:</span>
                        <button data-days="365" class="filter-preset px-3 py-1.5 text-sm rounded-full border border-gray-300 bg-white hover:bg-blue-50 transition">1 Year</button>
                        <button data-days="180" class="filter-preset px-3 py-1.5 text-sm rounded-full border border-gray-300 bg-white hover:bg-blue-50 transition">6 Months</button>
                        <button data-days="90" class="filter-preset px-3 py-1.5 text-sm rounded-full border border-gray-300 bg-white hover:bg-blue-50 transition active default-active">3 Months</button>
                        <button data-days="30" class="filter-preset px-3 py-1.5 text-sm rounded-full border border-gray-300 bg-white hover:bg-blue-50 transition">1 Month</button>
                        <div class="flex gap-1 items-center">
                            <input type="number" id="customDaysInput" placeholder="Days" value="30" class="w-20 px-2 py-1.5 text-sm border rounded-lg" min="1" step="1">
                            <button id="applyCustomDays" class="px-3 py-1.5 text-sm bg-gray-800 text-white rounded-lg hover:bg-gray-900">Apply</button>
                        </div>
                        <button id="resetFilter" class="px-3 py-1.5 text-sm bg-gray-200 rounded-full hover:bg-gray-300">Reset (Full)</button>
                    </div>
                    <div class="text-xs bg-gray-100 px-3 py-1 rounded-full" id="filterRangeLabel">📅 Last 90 days</div>
                </div>

                <div class="flex flex-wrap justify-between items-center gap-3 bg-white/70 p-3 rounded-xl shadow-md">
                    <div class="flex items-center gap-2">
                        <span class="font-semibold text-gray-700">🎨 Chart style:</span>
                        <select id="chartTypeSelect" class="bg-white border border-gray-300 rounded-lg px-3 py-1.5 text-sm focus:ring-blue-400">
                            <option value="line">📈 Price Line (Close)</option>
                            <option value="volume">📊 Volume (Bar)</option>
                            <option value="candlestick">🕯️ Candlestick (OHLC)</option>
                            <option value="ma20">📉 Price + MA(20) Trend</option>
                            <option value="scatter">💎 Price vs Volume Scatter</option>
                        </select>
                    </div>
                    <div class="text-xs text-gray-400 bg-gray-100 px-3 py-1 rounded-full">Interactive | Auto-update on filter</div>
                </div>

                <div class="chart-container bg-white shadow-xl border border-gray-100">
                    <div id="mainChart" style="height: 480px; width: 100%;"></div>
                </div>

                <div class="grid grid-cols-2 gap-4">
                    <div class="bg-white/80 rounded-xl p-3 shadow">
                        <p class="text-xs font-bold uppercase text-gray-500">📅 Best & worst days (filtered)</p>
                        <div class="flex justify-between text-sm mt-1"><span class="font-medium">🔥 Best gain:</span> <span id="bestGainDay" class="text-emerald-600">—</span></div>
                        <div class="flex justify-between text-sm"><span class="font-medium">📉 Worst loss:</span> <span id="worstLossDay" class="text-rose-600">—</span></div>
                        <div class="flex justify-between text-sm"><span class="font-medium">⚡ Trend slope (%/day):</span> <span id="trendSlope" class="font-mono">—</span></div>
                    </div>
                    <div class="bg-white/80 rounded-xl p-3 shadow">
                        <p class="text-xs font-bold uppercase text-gray-500">📐 Data coverage</p>
                        <div class="flex justify-between text-sm"><span>Trading days:</span> <span id="recordCount" class="font-bold">0</span></div>
                        <div class="flex justify-between text-sm"><span>Date range:</span> <span id="dateRange" class="text-xs">—</span></div>
                        <div class="flex justify-between text-sm"><span>Corr (Price/Volume):</span> <span id="priceVolCorr" class="font-mono">—</span></div>
                    </div>
                </div>
            </div>
        </div>
        <footer class="text-center text-[11px] text-gray-400 mt-8 border-t pt-5">Data from database | Last 5 years | Filters apply client-side</footer>
    </div>

    <script>
        // ---------- EMBEDDED DATA ----------
        const fullStockData = {data_json};
        console.log("Embedded data points:", fullStockData.length);
        if (fullStockData.length === 0) {{
            console.error("No data embedded! Check JSON generation.");
            document.body.innerHTML = '<div style="text-align:center; margin-top:50px;"><h2>Error: No data embedded in HTML file.</h2><p>Please regenerate the report.</p></div>';
        }}

        // Ensure each object has a dateObj and use dateStr
        for (let i = 0; i < fullStockData.length; i++) {{
            fullStockData[i].dateObj = new Date(fullStockData[i].dateStr);
            if (isNaN(fullStockData[i].dateObj.getTime())) {{
                console.warn("Invalid date at index", i, fullStockData[i].dateStr);
            }}
        }}
        fullStockData.sort((a,b) => a.dateObj - b.dateObj);
        console.log("Sorted data range:", fullStockData[0]?.dateStr, "to", fullStockData[fullStockData.length-1]?.dateStr);

        let filteredStockData = [];
        let currentFilterDays = 90;
        const DEFAULT_FILTER_DAYS = 90;

        // ----- Statistical functions -----
        function average(arr) {{ return arr.reduce((a,b)=>a+b,0) / (arr.length||1); }}
        function stdDev(arr) {{ let mean = average(arr); let sqDiff = arr.map(v=> (v-mean)**2); return Math.sqrt(average(sqDiff)); }}
        
        function pearsonCorrelation(x,y) {{
            let n = x.length;
            if(n<2) return 0;
            let sumX=0,sumY=0,sumXY=0,sumX2=0,sumY2=0;
            for(let i=0;i<n;i++) {{ sumX+=x[i]; sumY+=y[i]; sumXY+=x[i]*y[i]; sumX2+=x[i]*x[i]; sumY2+=y[i]*y[i]; }}
            let numerator = n*sumXY - sumX*sumY;
            let denominator = Math.sqrt((n*sumX2 - sumX*sumX) * (n*sumY2 - sumY*sumY));
            return denominator === 0 ? 0 : numerator/denominator;
        }}
        
        function computeCorrelationMatrix(dataArray, fields) {{
            const n = fields.length;
            let matrix = Array(n).fill().map(()=>Array(n).fill(0));
            let vectors = fields.map(f => dataArray.map(row => row[f]));
            for(let i=0;i<n;i++) {{
                for(let j=0;j<n;j++) {{
                    if(i===j) matrix[i][j]=1;
                    else matrix[i][j] = pearsonCorrelation(vectors[i], vectors[j]);
                }}
            }}
            return matrix;
        }}
        
        function analyzeAndRender(dataArray) {{
            if(!dataArray.length) {{
                document.getElementById('latestPrice').innerHTML = '—';
                document.getElementById('highPrice').innerHTML = '—';
                document.getElementById('lowPrice').innerHTML = '—';
                document.getElementById('avgPrice').innerHTML = '—';
                document.getElementById('totalReturn').innerHTML = '—';
                document.getElementById('volatility').innerHTML = '—';
                document.getElementById('avgVolume').innerHTML = '—';
                document.getElementById('totalVolume').innerHTML = '—';
                document.getElementById('recordCount').innerHTML = '0';
                document.getElementById('dateRange').innerHTML = '—';
                document.getElementById('trendSlope').innerHTML = '—';
                document.getElementById('bestGainDay').innerHTML = '—';
                document.getElementById('worstLossDay').innerHTML = '—';
                document.getElementById('priceVolCorr').innerHTML = '—';
                document.getElementById('insightText').innerHTML = '<p class="italic text-gray-400">No data for this filter. Try wider range.</p>';
                document.getElementById('topVolumeDays').innerHTML = '<li>—</li>';
                Plotly.react('correlationHeatmap', [], {{}});
                Plotly.react('mainChart', [{{x:[], y:[], type:'scatter'}}], {{title:"No data for selected period", annotations:[{{text:"Adjust filter", xref:"paper", yref:"paper", x:0.5, y:0.5, showarrow:false}}]}});
                return;
            }}
            
            const closes = dataArray.map(d=>d.price);
            const highs = dataArray.map(d=>d.maxp);
            const lows = dataArray.map(d=>d.minp);
            const opens = dataArray.map(d=>d.opnp);
            const volumes = dataArray.map(d=>d.qty);
            const firstClose = closes[0];
            const lastClose = closes[closes.length-1];
            const totalReturnPct = ((lastClose - firstClose)/firstClose)*100;
            
            let dailyReturns = [];
            for(let i=1;i<closes.length;i++) dailyReturns.push((closes[i]-closes[i-1])/closes[i-1]);
            const volatility = dailyReturns.length ? (stdDev(dailyReturns) * Math.sqrt(252) * 100) : 0;
            
            const n = closes.length;
            let sumX=0,sumY=0,sumXY=0,sumX2=0;
            for(let i=0;i<n;i++) {{ sumX+=i; sumY+=closes[i]; sumXY+=i*closes[i]; sumX2+=i*i; }}
            const slope = (n*sumXY - sumX*sumY) / (n*sumX2 - sumX*sumX);
            const slopePerDayPercent = (slope / firstClose) * 100;
            
            let maxGain=-Infinity, maxLoss=Infinity, bestDate='', worstDate='';
            for(let i=1;i<dataArray.length;i++) {{
                let pct = (closes[i]-closes[i-1])/closes[i-1]*100;
                if(pct>maxGain) {{ maxGain=pct; bestDate=dataArray[i].dateStr; }}
                if(pct<maxLoss) {{ maxLoss=pct; worstDate=dataArray[i].dateStr; }}
            }}
            
            let volList = dataArray.map(d=>({{date:d.dateStr, vol:d.qty}}));
            volList.sort((a,b)=>b.vol - a.vol);
            let top3 = volList.slice(0,3);
            
            const fields = ['price','maxp','minp','opnp','qty'];
            const corrMat = computeCorrelationMatrix(dataArray, fields);
            const priceVolCorr = corrMat[0][4];
            
            document.getElementById('latestPrice').innerHTML = lastClose.toFixed(2);
            document.getElementById('highPrice').innerHTML = Math.max(...closes).toFixed(2);
            document.getElementById('lowPrice').innerHTML = Math.min(...closes).toFixed(2);
            document.getElementById('avgPrice').innerHTML = average(closes).toFixed(2);
            const retClass = totalReturnPct >=0 ? 'text-emerald-600' : 'text-rose-600';
            document.getElementById('totalReturn').innerHTML = `<span class="${{retClass}}">${{totalReturnPct.toFixed(2)}}%</span>`;
            document.getElementById('volatility').innerHTML = volatility.toFixed(2)+'%';
            document.getElementById('avgVolume').innerHTML = Math.round(average(volumes)).toLocaleString();
            document.getElementById('totalVolume').innerHTML = volumes.reduce((a,b)=>a+b,0).toLocaleString();
            document.getElementById('recordCount').innerHTML = closes.length;
            document.getElementById('dateRange').innerHTML = `${{dataArray[0].dateStr}} → ${{dataArray[dataArray.length-1].dateStr}}`;
            document.getElementById('trendSlope').innerHTML = `${{slopePerDayPercent>0?'▲':'▼'}} ${{Math.abs(slopePerDayPercent).toFixed(3)}}%`;
            document.getElementById('bestGainDay').innerHTML = `${{maxGain.toFixed(2)}}% (${{bestDate||'—'}})`;
            document.getElementById('worstLossDay').innerHTML = `${{maxLoss.toFixed(2)}}% (${{worstDate||'—'}})`;
            document.getElementById('priceVolCorr').innerHTML = priceVolCorr.toFixed(3);
            
            let insightHtml = `<p class="text-sm"><span class="font-bold">📈 Trend:</span> ${{slopePerDayPercent>0?'Bullish slope':'Bearish slope'}} (${{Math.abs(slopePerDayPercent).toFixed(2)}}% avg daily change).</p>`;
            insightHtml += `<p class="text-sm"><span class="font-bold">🔄 Return:</span> ${{totalReturnPct.toFixed(2)}}% over ${{closes.length}} sessions.</p>`;
            insightHtml += `<p class="text-sm"><span class="font-bold">⚡ Volatility:</span> ${{volatility.toFixed(2)}}% (annualized).</p>`;
            insightHtml += `<p class="text-sm"><span class="font-bold">🧩 Price-Volume corr:</span> ${{priceVolCorr.toFixed(3)}} (${{Math.abs(priceVolCorr)>0.5?'significant':'weak'}})</p>`;
            document.getElementById('insightText').innerHTML = insightHtml;
            
            let topVolHtml = '';
            top3.forEach(v=>{{ topVolHtml += `<li>📆 ${{v.date}} → ${{v.vol.toLocaleString()}}</li>`; }});
            document.getElementById('topVolumeDays').innerHTML = topVolHtml || '<li>—</li>';
            
            drawCorrelationHeatmap(corrMat);
            const chartType = document.getElementById('chartTypeSelect').value;
            renderChart(chartType, dataArray);
        }}
        
        function drawCorrelationHeatmap(corrMatrix) {{
            const fields = ['price','maxp','minp','opnp','qty'];
            let data = [{{
                z: corrMatrix,
                x: fields, y: fields,
                type: 'heatmap', colorscale: 'RdBu', zmin: -1, zmax: 1,
                text: corrMatrix.map(row=>row.map(v=>v.toFixed(2))),
                texttemplate: '%{{text}}', hoverongaps: false, showscale: true
            }}];
            let layout = {{ height: 260, margin: {{ t: 30, b: 30, l: 50, r: 20 }}, xaxis: {{ tickangle: -25 }}, yaxis: {{ autorange: 'reversed' }}, font: {{ size: 10 }} }};
            Plotly.react('correlationHeatmap', data, layout, {{ responsive: true, displayModeBar: false }});
        }}
        
        function renderChart(type, dataArray) {{
            if(!dataArray.length) return;
            const dates = dataArray.map(d=>d.dateStr);
            const closes = dataArray.map(d=>d.price);
            const volumes = dataArray.map(d=>d.qty);
            const opens = dataArray.map(d=>d.opnp);
            const highs = dataArray.map(d=>d.maxp);
            const lows = dataArray.map(d=>d.minp);
            
            let chartData = [];
            let layout = {{ title: {{ text: '', font: {{ size: 14 }} }}, xaxis: {{ title: 'Trade Date', tickangle: -40, type: 'category' }}, yaxis: {{ title: 'Price (THB)' }}, template: 'plotly_white', hovermode: 'x unified', margin: {{ t: 40, b: 60, l: 50, r: 30 }}, legend: {{ orientation: 'h', yanchor: 'bottom', y: 1.02 }} }};
            
            switch(type) {{
                case 'line':
                    chartData = [{{ x: dates, y: closes, mode: 'lines', name: 'Close Price', line: {{ color: '#2563eb', width: 2.5 }}, fill: 'tozeroy', fillcolor: 'rgba(37,99,235,0.05)' }}];
                    break;
                case 'volume':
                    let colors = closes.map((c,i)=> (i>0 && c>=closes[i-1]) ? '#22c55e' : '#ef4444');
                    chartData = [{{ x: dates, y: volumes, type: 'bar', name: 'Volume', marker: {{ color: colors, opacity: 0.7 }}, yaxis: 'y' }}];
                    layout.yaxis = {{ title: 'Volume (shares)', tickformat: ',.0f' }};
                    layout.title.text = 'Daily Trading Volume';
                    break;
                case 'candlestick':
                    chartData = [{{ x: dates, open: opens, high: highs, low: lows, close: closes, type: 'candlestick', name: 'OHLC', increasing: {{ line: {{ color: '#2ecc71' }} }}, decreasing: {{ line: {{ color: '#e74c3c' }} }} }}];
                    layout.yaxis.title = 'Price (THB)';
                    layout.title.text = 'Candlestick Chart';
                    break;
                case 'ma20':
                    let ma20 = [];
                    for(let i=0;i<closes.length;i++) {{
                        if(i<19) ma20.push(null);
                        else {{ let sum=0; for(let j=i-19;j<=i;j++) sum+=closes[j]; ma20.push(sum/20); }}
                    }}
                    chartData = [
                        {{ x: dates, y: closes, mode: 'lines', name: 'Close', line: {{ color: '#1f2937', width: 2 }} }},
                        {{ x: dates, y: ma20, mode: 'lines', name: 'SMA(20)', line: {{ color: '#f97316', width: 2.5, dash: 'dot' }} }}
                    ];
                    break;
                case 'scatter':
                    chartData = [{{ x: volumes, y: closes, mode: 'markers', type: 'scatter', marker: {{ color: '#3b82f6', size: 6, opacity: 0.6 }}, text: dates, hovertemplate: 'Vol: %{{x:,}}<br>Price: %{{y}}<br>%{{text}}<extra></extra>' }}];
                    layout.xaxis = {{ title: 'Volume (shares)', tickformat: ',.0f' }};
                    layout.yaxis = {{ title: 'Close Price (THB)' }};
                    layout.title.text = 'Price vs Volume Scatter';
                    break;
                default: chartData = [{{ x: dates, y: closes, mode: 'lines', name: 'Close' }}];
            }}
            Plotly.react('mainChart', chartData, layout, {{ responsive: true, displayModeBar: true }});
        }}
        
        function filterDataByDays(daysFromLatest) {{
            if(daysFromLatest === null) return [...fullStockData];
            const latestDate = fullStockData[fullStockData.length-1].dateObj;
            const cutoffDate = new Date(latestDate);
            cutoffDate.setDate(cutoffDate.getDate() - daysFromLatest);
            return fullStockData.filter(row => row.dateObj >= cutoffDate);
        }}
        
        function applyFilter(days) {{
            currentFilterDays = days;
            const filtered = filterDataByDays(days);
            filteredStockData = filtered;
            const rangeLabel = document.getElementById('filterRangeLabel');
            if(days === null) rangeLabel.innerHTML = '📅 Full history';
            else rangeLabel.innerHTML = `📅 Last ${{days}} days (${{filtered.length}} days)`;
            analyzeAndRender(filteredStockData);
            document.querySelectorAll('.filter-preset').forEach(btn => btn.classList.remove('active', 'bg-blue-600', 'text-white', 'border-blue-600'));
            if(days !== null) {{
                const activeBtn = document.querySelector(`.filter-preset[data-days="${{days}}"]`);
                if(activeBtn) activeBtn.classList.add('active', 'bg-blue-600', 'text-white', 'border-blue-600');
            }}
        }}
        
        function resetFilter() {{ applyFilter(null); }}
        
        document.querySelectorAll('.filter-preset').forEach(btn => {{
            btn.addEventListener('click', (e) => {{
                const days = parseInt(btn.getAttribute('data-days'), 10);
                if(!isNaN(days)) applyFilter(days);
            }});
        }});
        document.getElementById('resetFilter').addEventListener('click', resetFilter);
        document.getElementById('applyCustomDays').addEventListener('click', () => {{
            let daysInput = document.getElementById('customDaysInput');
            let days = parseInt(daysInput.value, 10);
            if (isNaN(days) || days < 1) {{
                days = 30;
                daysInput.value = 30;
            }}
            applyFilter(days);
        }});
        document.getElementById('chartTypeSelect').addEventListener('change', (e) => {{
            if(filteredStockData.length) renderChart(e.target.value, filteredStockData);
        }});
        
        function init() {{
            document.getElementById('customDaysInput').value = 30;
            const defaultBtn = document.querySelector('.filter-preset[data-days="90"]');
            if(defaultBtn) defaultBtn.classList.add('active', 'bg-blue-600', 'text-white', 'border-blue-600');
            document.getElementById('filterRangeLabel').innerHTML = '📅 Last 90 days (default)';
            if (fullStockData.length > 0) {{
                applyFilter(DEFAULT_FILTER_DAYS);
            }} else {{
                document.getElementById('insightText').innerHTML = '<p class="italic text-gray-400">No data embedded. Please regenerate the report.</p>';
            }}
        }}
        init();
    </script>
</body>
</html>"""

# Format the template with the symbol and data
final_html = html_template.format(symbol=name, data_json=data_json_str)

# Write the HTML file
#output_path = Path.cwd() / f"{name}.html"
output_path = os.path.join(csv_path, f"{name}.html")
with open(output_path, "w", encoding="utf-8") as f:
    f.write(final_html)

print(f"✅ HTML report generated: {output_path}")
print(f"File size: {output_path.stat().st_size / 1024:.2f} KB")
print(f"Double-click the file to open in your browser. If still no data, open browser console (F12) to see error messages.")

Data points to embed: 1207
✅ HTML report generated: C:\Users\PC1\OneDrive\A4\CSV\PTT.html


AttributeError: 'str' object has no attribute 'stat'